# Считывание Данных

In [ ]:
import pandas as pd 
import polars as pl
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive_files = '/kaggle/input/datasets/shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021/*.parquet'
drives_needed_cols = ["request_datetime", "PULocationID"]
drives = pl.scan_parquet(drive_files).select([pl.col(x) for x in drives_needed_cols])
print(drives.head(5).collect())

In [ ]:
meteodata_files = '/kaggle/input/datasets/shuhengmo/uber-nyc-forhire-vehicles-trip-data-2021/nyc 2021-01-01 to 2021-12-31.csv'
meteo_needed_cols = [
    'datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob',
    'preciptype', 'snow', 'snowdepth', 'windgust','windspeed', 'winddir',
    'sealevelpressure', 'cloudcover', 'visibility', 'uvindex'
]
meteodata = pl.scan_csv(meteodata_files).select([pl.col(x) for x in meteo_needed_cols])
print(meteodata.head(5).collect())

# Первичная обработка данных

## Сжимаем датасет до часовой точности, суммируя поездки по локациям

In [ ]:
start_date = datetime(2021, 1, 1)
second_week = datetime(2021, 1, 7)

drives_lazy = (
    drives
    .filter(pl.col("request_datetime") >= start_date)
    # 1. Сначала агрегируем только по ключевым полям
    .group_by([
        pl.col("request_datetime").dt.truncate("1h").alias("hour_bucket"),
        "PULocationID"
    ])
    .agg(pl.len().alias("trips_count"))
    
    # 2. Сортируем сжатый датасет
    .sort(["PULocationID", "hour_bucket"])
)

print(drives_lazy.collect(engine="streaming").head(5))

## Заполняем временной ряд полностью, добавляя записи даже в которые не произошло ни одной поездки

In [ ]:
end_date = drives_lazy.select(pl.col("hour_bucket").max()).collect().item()
target_dtype = drives_lazy.collect_schema()["hour_bucket"] # Получаем тип данных (например, Datetime('us'))
time_grid = pl.datetime_range(
    start=start_date, 
    end=end_date, 
    interval="1h", 
    eager=True
).alias("hour_bucket").to_frame().cast({"hour_bucket": target_dtype}).lazy()

locations = drives.select(pl.col("PULocationID").unique())

grid = time_grid.join(locations, how="cross")

full_drives_lazy = grid.join(
    drives_lazy, 
    on=["hour_bucket", "PULocationID"], 
    how="left"
).with_columns(
    pl.col("trips_count").fill_null(0) # Теперь дырок во времени нет
)

full_drives_lazy = full_drives_lazy.sort(["PULocationID", "hour_bucket"])

print(full_drives_lazy.collect(engine="streaming").head(6))

## Разделение request_datetime на hour, dayofweek, month, day

In [ ]:
full_drives_lazy = (
    full_drives_lazy
    # 1. Достаем признаки времени из компактного hour_bucket
    .with_columns([
        pl.col("hour_bucket").dt.month().alias("month"),
        pl.col("hour_bucket").dt.day().alias("day"),
        pl.col("hour_bucket").dt.weekday().alias("dayofweek"),
        pl.col("hour_bucket").dt.hour().alias("hour"),
        pl.col("hour_bucket").dt.date().alias("date"),
    ])
    
    # 2. Считаем лаги на агрегированных данных
    .with_columns([
        pl.col("trips_count").shift(1).over("PULocationID").fill_null(0).alias("trips_lag_1h"),
        pl.col("trips_count").shift(24).over("PULocationID").fill_null(0).alias("trips_lag_day"),
        pl.col("trips_count").shift(24 * 7).over("PULocationID").fill_null(0).alias("trips_lag_week"),
    ])
    
    # 3. Отсекаем ненужный хвост (теперь это быстро)
    .filter(pl.col("hour_bucket") >= second_week)   
)

print(full_drives_lazy.head(5).collect())

## Преобразование временных данных в метеоданных

In [ ]:
meteodata_lazy = (
    meteodata.with_columns(
        pl.col('datetime').str.to_date().alias('date')
    )
    .drop('datetime')
)

print(meteodata_lazy.head(5).collect())

## Join метеоданных 

In [ ]:
result_lazy = full_drives_lazy.join(meteodata_lazy, on="date", how="inner")
result_lazy = result_lazy.drop("date")
result_df = result_lazy.collect()
result_df.head(25)

## Заполнение пропусков и кодирование типа осадков

In [ ]:
result_df = (
    result_df.
    with_columns([
        pl.col("preciptype").str.contains("rain").fill_null(False).cast(pl.Int8).alias("is_rain"),
        pl.col("preciptype").str.contains("snow").fill_null(False).cast(pl.Int8).alias("is_snow")
    ])
    .drop("preciptype")
)
result_df.head(25)

## Аналитика

In [ ]:
cols_for_base_analysis = [
    "feelslike", "dew", "humidity",
    "snowdepth", "winddir", "sealevelpressure",
    "cloudcover", "visibility", "uvindex"
]

base_atributes = ["temp", "precip", "precipprob", "windgust", "windspeed"]

target = "trips_count"


description = {
    "hour_bucket": "Дата и время с точностью до часа",
    "PULocationID": "Зона такси TLC, в которой начался рейс",
    "trips_count": "Кол-во поездок за час из определенной зоны",
    "trips_lag_1h": "Кол-во поездок за час из определенной зоны за час до текущего времени",
    "trips_lag_day": "Кол-во поездок за сутки из определенной зоны за час до текущего времени",
    "trips_lag_week": "Кол-во поездок за неделю из определенной зоны за час до текущего времени",
    "hour": "Час вызова такси",
    "dayofweek": "День недели вызова такси(1-7)",
    "month": "Месяц вызова такси",
    "day": "День вызова такси",
    "temp": "Температура воздуха",
    "feelslike": "Ощущаемая температура воздуха",
    "dew": "Точка росы",
    "humidity": "Влажность",
    "precip": "Осадки?",
    "precipprob": "Вероятность осадков",
    "snowdepth": "Глубина снега",
    "windgust": "Порывы ветра",
    "windspeed": "Скорость ветра",
    "winddir": "Направление ветра",
    "sealevelpressure": "Атмосферное давление, приведенное к среднему уровню моря ",
    "cloudcover": "Облачность",
    "visibility": "Видимость",
    "uvindex": "УФ-Индекс"
}

## Анализ выбросов. Скользящее окно

In [ ]:
window_size = 24 # Размер окна (например, 5 дней)
threshold = 3    # Коэффициент чувствительности
df_processed = result_df.with_columns([
    # 1. Считаем границы во Float64 (это неизбежно для математики)
    (pl.col(target).cast(pl.Float64).rolling_median(window_size=window_size, center=True)).alias("med"),
    (pl.col(target).cast(pl.Float64).rolling_std(window_size=window_size, center=True)).alias("std")
]).with_columns([
    # 2. Ограничиваем нижнюю границу нулем, чтобы она была логичной (поездки >= 0)
    (pl.col("med") + threshold * pl.col("std")).alias("upper"),
    (pl.col("med") - threshold * pl.col("std")).clip(lower_bound=0).alias("lower") 
]).with_columns([
    # 3. Срезаем выбросы и возвращаем в целое число (u32)
    pl.col(target).cast(pl.Float64)
      .clip(pl.col("lower"), pl.col("upper"))
      .round(0) # Округляем до ближайшего целого
      .cast(pl.UInt32) # Возвращаем в натуральный вид
      .alias("value_cleaned")
])

result_df = (
    df_processed.
    drop(["med", "std", "upper", "lower", "value_cleaned"])
)
result_df.head(43)

In [ ]:
stats = []

for col in cols_for_base_analysis:
    stats.extend([
        pl.col(col).min().alias(f'{col}_min'),
        pl.col(col).max().alias(f'{col}_max'),
        pl.col(col).mean().alias(f'{col}_mean'),
        pl.col(col).median().alias(f'{col}_median'),
        pl.col(col).std().alias(f'{col}_std')
    ])

results = result_df.select(stats)
for col in cols_for_base_analysis:
    print(f"--- {col} ---")
    print(f"Описание: {description[col]}")
    print(f"Мин: {results[0, f'{col}_min']}")
    print(f"Макс: {results[0, f'{col}_max']}")
    print(f"Среднее: {results[0, f'{col}_mean']:.2f}")
    print(f"Медиана: {results[0, f'{col}_median']}")
    print(f"Ст. отк.: {results[0, f'{col}_std']:.2f}\n")

## Распределение основных признаков

In [ ]:
test_flag = True

In [ ]:
if not test_flag:
    stats = []

    for col in base_atributes:
        stats.extend([
            pl.col(col).min().alias(f'{col}_min'),
            pl.col(col).max().alias(f'{col}_max'),
            pl.col(col).mean().alias(f'{col}_mean'),
            pl.col(col).median().alias(f'{col}_median'),
            pl.col(col).std().alias(f'{col}_std')
        ])

    results = result_df.select(stats)
    for col in base_atributes:
        print(f"--- {col} ---")
        print(f"Описание: {description[col]}")
        print(f"Мин: {results[0, f'{col}_min']}")
        print(f"Макс: {results[0, f'{col}_max']}")
        print(f"Среднее: {results[0, f'{col}_mean']:.2f}")
        print(f"Медиана: {results[0, f'{col}_median']}")
        print(f"Ст. отк.: {results[0, f'{col}_std']:.2f}\n")

        plt.figure(figsize=(10, 6))
        # Гистограмма распределения
        sns.histplot(result_df[col], bins=100, kde=True, color='blue')

        plt.xlabel(description[col])
        plt.ylabel('Частота')
        # Ограничим X, если есть экстремальные выбросы (например, аэропорты)
        plt.xlim(right=result_df[col].quantile(0.99))
        plt.show()

## Распределение Целевой переменной

In [ ]:
if not test_flag:
    stats = []
    stats.extend([
        pl.col(target).min().alias(f'{target}_min'),
        pl.col(target).max().alias(f'{target}_max'),
        pl.col(target).mean().alias(f'{target}_mean'),
        pl.col(target).median().alias(f'{target}_median'),
        pl.col(target).std().alias(f'{target}_std')
    ])
    results = result_df.select(stats)
    print(f"--- {target} ---")
    print(f"Описание: {description[target]}")
    print(f"Мин: {results[0, f'{target}_min']}")
    print(f"Макс: {results[0, f'{target}_max']}")
    print(f"Среднее: {results[0, f'{target}_mean']:.2f}")
    print(f"Медиана: {results[0, f'{target}_median']}")
    print(f"Ст. отк.: {results[0, f'{target}_std']:.2f}\n")

    plt.figure(figsize=(10, 6))
    # Гистограмма распределения
    sns.histplot(result_df['trips_count'], bins=100, kde=True, color='blue')

    plt.title('Распределение количества поездок в час')
    plt.xlabel('Количество поездок')
    plt.ylabel('Частота')
    plt.show()

## Разделяем датасет на выборку для кросс-валидации и тестовую

In [ ]:
# 1. Сортируем (на всякий случай, для временных рядов это критично)
result_df = result_df.sort("hour_bucket")

# 2. Разделяем по датам (например, 2021 год)
train_cv_finish = datetime(2021, 11, 30)
train_cv = result_df.filter(pl.col("hour_bucket") < train_cv_finish)
test  = result_df.filter(pl.col("hour_bucket") >= train_cv_finish)

print(f"Train: {train_cv.shape[0]} строк")
print(f"Test:  {test.shape[0]} строк")

## Масштабируем метеоданные

In [ ]:
# from sklearn.preprocessing import MinMaxScaler
# # 1. Выделяем числовые признаки
# numeric_cols = [
#     'datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipprob',
#     'snowdepth', 'windgust','windspeed', 'winddir',
#     'sealevelpressure', 'cloudcover', 'visibility', 'uvindex'
# ]
# # 2. Инициализируем MinMaxScaler
# # Он идеально подходит для LSTM
# scaler = MinMaxScaler(feature_range=(0, 1))
# # 3. Применяем (только после разделения на train/test!)
# train_scaled = scaler.fit_transform(X_train[numeric_cols])
# test_scaled = scaler.transform(X_test[numeric_cols])

# Построение моделей

## Алгоритм Random Forest
### Подбор гиперпараметров алгоритма

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import pandas as pd
import optuna
import numpy as np
import warnings

# 1. Подготовка признаков
features = [col for col in train_cv.columns if col not in [target, "hour_bucket"]]

# 2. ПРЕВРАЩАЕМ В NUMPY СРАЗУ (Чтобы не было имен колонок вообще)
X_np = train_cv.select(features).to_pandas().values
y_np = train_cv.select(target).to_pandas().values.ravel()

# 31 день по 24 часа — хороший размер для валидации
tscv = TimeSeriesSplit(n_splits=3, test_size=24*31)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [ ]:
def objective(trial):
    param = {
    "boosting_type": "rf",
    "objective": "regression_l1",
    "metric": "mae",
    "n_estimators": 1189, 
    "num_leaves": 3759,
    "max_depth": 29,
    "min_child_samples": 247,
    "subsample": 0.9643311966144387,
    "subsample_freq": 1,
    "feature_fraction":  0.47817161713615186,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
    "device": "gpu"
}
    
    scores = []
    
    # Теперь итерируемся по X_np (массиву), так чище и быстрее
    for i, (train_idx, val_idx) in enumerate(tscv.split(X_np)):
        X_train, X_val = X_np[train_idx], X_np[val_idx]
        y_train, y_val = y_np[train_idx], y_np[val_idx]

        model = lgb.LGBMRegressor(**param)
        
        # Здесь нет имен колонок, LightGBM будет работать только с индексами
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        
        mae = mean_absolute_error(y_val, preds)
        scores.append(mae)

        trial.report(mae, i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

# Настройка исследования
study = optuna.create_study(
    direction="minimize", 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

# Запуск
study.optimize(objective, n_trials=20)

print(f"Лучшие параметры: {study.best_params}")

что-то до PI

In [ ]:
import optuna.visualization as vis

#vis.plot_param_importances(study)

vis.plot_parallel_coordinate(study)

### Отбор признаков с помощью Permutation Importance

In [ ]:
# best_params = study.best_params

best_params = {
    "n_estimators": 300,
    "min_child_samples": 200,
    "subsample": 0.7,
    "feature_fraction": 0.8
}


final_params_rf = {
    "boosting_type": "rf",
    "objective": "regression_l1",
    "metric": "mae",
    "n_estimators": 1189, 
    "num_leaves": 3759,
    "max_depth": 29,
    "min_child_samples": 247,
    "subsample": 0.9643311966144387,
    "subsample_freq": 1,
    "feature_fraction":  0.47817161713615186,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
}

final_params_bstg = {
    **best_params,
    "boosting_type": "gbdt",  
    "objective": "regression",
    "metric": "rmse",

    "num_leaves": 64,         
    "max_depth": -1,           

    "learning_rate": 0.05,   
    "n_estimators": 1000,      

    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1
}

data_limit = 500000
X_np = X_np[:data_limit]
y_np = y_np[:data_limit]

model = lgb.LGBMRegressor(**final_params_bstg)
model.fit(X_np, y_np)

preds = model.predict(X_np)
rmse = np.sqrt(mean_squared_error(y_np, preds))
print("RMSE:", rmse)

In [ ]:
# from sklearn.inspection import permutation_importance
#
# # best_params = study.best_params
#
# result = permutation_importance(
#     model,
#     pd.DataFrame(X_np, columns=features),
#     y_np,
#     n_repeats=5,
#     random_state=42,
#     n_jobs=-1
# )
#
# perm_df = pd.DataFrame({
#     "feature": features,
#     "importance": result.importances_mean
# }).sort_values(by="importance", ascending=False)
#
# print(perm_df.head(20))

Удаляем бесполезные признаки

In [ ]:
# from sklearn.inspection import permutation_importance
# import numpy as np
#
# # 1. Начинаем со всех признаков
# k = 20
# current_features = perm_df.head(k)["feature"].tolist()
# history = []
#
# step = 2  # сколько признаков удаляем за итерацию (можно 1 или 2)
#
# while len(current_features) > 5:
#     # 1. Готовим данные с текущими фичами
#     X_curr = df_model[current_features].values
#
#     # 2. Обучаем модель ОДИН раз на текущих данных, чтобы узнать важность
#     model.fit(X_curr, y_np)
#
#     # 3. Считаем важность (например, через встроенный feature_importances_)
#     # Или через permutation_importance, но это дольше
#     importances = model.feature_importances_
#
#     # 4. Создаем таблицу важности и сортируем
#     current_importance_df = pd.DataFrame({
#         'feature': current_features,
#         'imp': importances
#     }).sort_values('imp', ascending=False)
#
#     # 5. Оставляем только ТОП признаков
#     current_features = current_importance_df.head(len(current_features) - step)['feature'].tolist()
#
#
#
#
#     # 3. CV
#     scores = []
#     for train_idx, val_idx in tscv.split(X_np_current):
#         X_train, X_val = X_np_current[train_idx], X_np_current[val_idx]
#         y_train, y_val = y_np[train_idx], y_np[val_idx]
#
#         model = lgb.LGBMRegressor(**final_params_bstg)
#         model.fit(X_train, y_train)
#
#         preds = model.predict(X_val)
#         rmse = np.sqrt(mean_squared_error(y_val, preds))
#         scores.append(rmse)
#
#     mean_rmse = np.mean(scores)
#     print("CV RMSE:", mean_rmse)
#
#     # сохраняем историю
#     history.append((len(current_features), mean_rmse, current_features.copy()))
#
#     # 5. Удаляем худшие признаки
#     to_remove = current_features[k:k-step-1:-1]
#     k -= step
#     print("Удаляем:", to_remove)
#
#     current_features = [f for f in current_features if f not in to_remove]
#
#
# # Итог
# print("\n\n=== ЛУЧШИЕ РЕЗУЛЬТАТЫ ===")
# history_sorted = sorted(history, key=lambda x: x[1])
#
# for n_feats, rmse, feats in history_sorted[:5]:
#     print(f"Фичей: {n_feats}, RMSE: {rmse}")
#     print(feats)
#     print("-" * 40)

## Feature Importance

In [ ]:
# import pandas as pd
# import numpy as np
# import lightgbm as lgb
# from sklearn.metrics import mean_squared_error
# from sklearn.model_selection import train_test_split
#
# # --- ПОДГОТОВКА ДАННЫХ ---
# # Если у вас есть список названий колонок (например, features_list), укажите его здесь
# # Если данные из pandas DataFrame, используйте X.columns
# column_names = getattr(X_np, 'columns', [f"f_{i}" for i in range(X_np.shape[1])])
#
# # Принудительно создаем DataFrame с именами колонок
# X_df = pd.DataFrame(X_np, columns=column_names)
#
# X_train, X_val, y_train, y_val = train_test_split(X_df, y_np, test_size=0.2, random_state=42)
#
# def rfe_with_lgbm(X_tr, X_v, y_tr, y_v, params, n_to_keep):
#     # Работаем с копиями, чтобы не испортить оригиналы
#     current_X_tr = X_tr.copy()
#     current_X_v = X_v.copy()
#
#     print(f"{'Осталось':<10} | {'RMSE (Val)':<12} | {'Удаленный признак'}")
#     print("-" * 60)
#
#     while current_X_tr.shape[1] > n_to_keep:
#         # 1. Обучаем модель
#         m = lgb.LGBMRegressor(**params)
#         m.fit(current_X_tr, y_tr)
#
#         # 2. Считаем метрику
#         v_preds = m.predict(current_X_v)
#         score = np.sqrt(mean_squared_error(y_v, v_preds))
#
#         # 3. Важность признаков (используем 'gain', так как для регрессии он чувствительнее)
#         # Если хотите оставить старый способ, замените на m.feature_importances_
#         importances = pd.Series(
#             m.booster_.feature_importance(importance_type='gain'),
#             index=current_X_tr.columns
#         )
#
#         # Находим название признака с минимальной важностью
#         worst_feat = importances.idxmin()
#
#         print(f"{current_X_tr.shape[1]:<10} | {score:<12.4f} | {worst_feat}")
#а 
#         # 4. Удаляем колонку по названию
#         current_X_tr.drop(columns=[worst_feat], inplace=True)
#         current_X_v.drop(columns=[worst_feat], inplace=True)
#
#     # Финальный расчет
#     m.fit(current_X_tr, y_tr)
#     final_score = np.sqrt(mean_squared_error(y_v, m.predict(current_X_v)))
#     print(f"{current_X_tr.shape[1]:<10} | {final_score:<12.4f} | --- ГОТОВО ---")
#
#     return current_X_tr.columns.tolist()
#
# # Запуск
# best_features = rfe_with_lgbm(X_train, X_val, y_train, y_val, final_params_bstg, n_to_keep=2)

## Permutation Importance

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

# 1. ОБЯЗАТЕЛЬНО: укажите названия колонок вручную, если X_np - это массив
# Пример: feature_names = list(your_original_df.columns)
# feature_names = [f"Признак_{i}" for i in range(X_np.shape[1])] # Замените на свои названия

X_df = pd.DataFrame(X_np, columns=features)
X_train, X_val, y_train, y_val = train_test_split(X_df, y_np, test_size=0.2, random_state=42)

def rfe_with_permutation(X_tr, X_v, y_tr, y_v, params, n_to_keep):
    current_X_tr = X_tr.copy()
    current_X_v = X_v.copy()

    print(f"{'Осталось':<10} | {'RMSE (Val)':<12} | {'Удаляемый (PI)'}")
    print("-" * 55)

    while current_X_tr.shape[1] > n_to_keep:
        m = lgb.LGBMRegressor(**params)
        m.fit(current_X_tr, y_tr)

        # Считаем RMSE
        score = np.sqrt(mean_squared_error(y_v, m.predict(current_X_v)))

        # Считаем Permutation Importance на ВАЛИДАЦИОННЫХ данных
        # n_repeats=3 достаточно для быстрой оценки
        r = permutation_importance(m, current_X_v, y_v, n_repeats=3, random_state=42, n_jobs=-1)

        # Находим самый бесполезный (с минимальным влиянием на RMSE)
        pi_importances = pd.Series(r.importances_mean, index=current_X_tr.columns)
        worst_feat = pi_importances.idxmin()

        print(f"{current_X_tr.shape[1]:<10} | {score:<12.4f} | {worst_feat}")

        current_X_tr.drop(columns=[worst_feat], inplace=True)
        current_X_v.drop(columns=[worst_feat], inplace=True)

    return current_X_tr.columns.tolist()

# Запуск
best_features = rfe_with_permutation(X_train, X_val, y_train, y_val, final_params_bstg, n_to_keep=5)

## Алгоритм XGBoost
### Подбор гиперпараметров алгоритма

### Отбор признаков с помощью Permitation Importance

### Отбор признаков с помощью XGBoost

### Обучение модели

## LSTM

### Подготовка данных для LSTM. Масштабирование метеоданных

### Обучение модели

# Оценка построенных моделей на отложенной выборке